## Odessey

**Odessey** is a simple migration simulation based on the gravity model, where each cell in the grid maintains a list of occupants. At every timestep, an individual has a probability of moving, with their destination determined by a weighted roulette wheel selection based on the normalised inverse square distance to other cells

*Scroll down to the bottom of the workbook untill you can see the display and then click on* "**Run all**" *above*

In [1]:
import random
import math
import ipywidgets as widgets
from IPython.display import display, HTML
import time

### This is just the display code

In [2]:
class SVGCanvas:
    def __init__(self,width,height,size):
        self.width = width
        self.height = height
        self.size = size
        self.html_s =""

    def clear(self):
        self.html_s = f'<svg width="{self.width}" height="{self.height}"style="border:1px solid black">'

    def addCircle(self,x_pos,y_pos,color):
        x_pos = x_pos/self.size
        y_pos = y_pos/self.size
        r = 1/(self.size*2)
        int_x = int((x_pos*self.width)+(r*self.width))
        int_y = int((y_pos*self.height)+(r*self.width))
        int_r = int(r*self.width)
        self.html_s +=f'<circle cx="{int_x}" cy="{int_y}" r="{int_r}" fill="{color}"/>'

    def addRect(self,x_pos,y_pos,color):
        x_pos = x_pos/self.size
        y_pos = y_pos/self.size
        height = 1/self.size
        width = 1/self.size
        int_x = str(int(x_pos*self.width))
        int_y = str(int(y_pos*self.height))
        int_width = str(int(width*self.width))
        int_height = str(int(height*self.height))
        self.html_s +=f'<rect x="{int_x}" y="{int_y}" width="{int_width}" height="{int_height}" fill="{color}"/>'

    def getCanvas(self):
        self.html_s += "</svg>"
        return self.html_s

In [3]:
# Constants
MEAN_POP = 3000
STD_DEV_POP = 20
GRID_SIZE = 32
MAX_ITERATIONS = 100
PROB_MOVE = 0.1
DESTINATION = False

### init grid
**init_grid** creates two 3D arrays. The first two dimensions in both arrays are row/columns of a spatial grid. The third dimension of array **grid** is a list of occupants identified by their position in the population array (see below). The third dimension of the **distance** array is the inverse squared distance of the cell from all of the other cells

In [4]:
def init_grid(grid,distance, size):

    def inverse_sqr_distance(y1, x1, y2, x2):
        dx = x2 - x1
        dy = y2 - y1
        dist_sq = dx * dx + dy * dy
        if dist_sq == 0:
            return 0.0
        return 1.0 / dist_sq

    for row in range(size):
        grid.append([])
        distance.append([])
        for col in range(size):
            grid[row].append([])
            dist_row = []
            for y in range(size):
                for x in range(size):
                    d  = inverse_sqr_distance(row,col,y,x)
                    dist_row.append(d)
            distance[row].append(dist_row)

### init_population

**init_population creates** a population table; each row has an ID and the row and column that the person occupies in the grid.  
*Note that in this version all of the population is created in the central cell of the **grid.***

In [5]:
def init_population(population, grid):
    c_max = len(grid)
    r_max = len(grid[0])
    pop_count = 0 # count number in population
    for row in range(r_max):   #row loop
        for col in range(c_max):   # col loop
            act_pop = 0
            if row == int(r_max/2) and col == int(c_max/2):
                act_pop = int(random.gauss(MEAN_POP, STD_DEV_POP))
            for _ in range(act_pop): # person loop
                grid[row][col].append(pop_count)
                population.append([pop_count,row,col])
                pop_count = pop_count + 1

### move
**move** simulates individual movement across **grid** by calculating weighted probabilities based on **distance**, then selecting which occupants in a given cell will relocate according to a configurable movement probability. It returns a list of movers with their original indices and new target coordinates for subsequent grid updates.

In [6]:
def move(row, col, grid, distance,destination):

    def move_to(y,x):
        if destination:
            if x!=5 or y!=5:
                if random.random()<0.1:
                    return 5,5
        choice = random.random()
        for i in range(len(roulette_wheel)):
            if choice <= roulette_wheel[i]:
                    return i // GRID_SIZE, i % GRID_SIZE
        return 0, 0  # Fallback (should not happen)

    # Calculate roulette wheel weights
    r_max = len(grid)
    c_max = len(grid[0])

    weights = distance[row][col]
    total_weight = sum(weights)

    roulette_wheel = []
    accumulate = 0.0
    for i in range(len(weights)):
        if total_weight > 0:
            accumulate = accumulate + (weights[i] / total_weight)
        roulette_wheel.append(accumulate)

    movers = []
    cell_pop = grid[row][col]
    for p in range(len(cell_pop)):
        if random.random()<PROB_MOVE:
            p_idx = cell_pop[p]
            nr, nc = move_to(row,col)
            movers.append([p_idx, nr, nc])

    return movers

### migrate
**migrate** tests each cell of **grid** in turn and determines which of the occupants will move and where to. Having obtained a list of all the occupants and where they are moving to, **grid** and **population** are updated.


In [7]:
def migrate(grid, population,distance,destination):
    movers = []
    r_max = len(grid)
    c_max = len(grid[0])
    movers = []
    # Collect all movers
    for row in range(r_max):
        for col in range(c_max):
            movers+=move(row, col, grid,distance,destination)

    # Execute moves
    for i in range(len(movers)):
        m = movers[i]
        p_id = m[0]  # This is the actual index in population array
        new_r = m[1]
        new_c = m[2]


        p = population[p_id]
        old_r = p[1]
        old_c = p[2]

        # Skip if moving to same cell
        if old_r == new_r and old_c == new_c:
            continue

        # Update population record
        p[1] = new_r
        p[2] = new_c

        # Remove from old cell
        old_cell = grid[old_r][old_c]
        old_cell.remove(p_id)
        grid[new_r][new_c].append(p_id)

    return len(movers)

In [8]:
def draw_grid(svg,grid,population):
    r_max = len(grid)
    c_max = len(grid[0])
    for row in range(r_max):
        for col in range(c_max):
            pop = min(255,len(grid[row][col]))
            r_col = 255-pop
            if pop ==0:
                svg.addRect(row,col,f'rgb(200,220,200)')
            else:
                svg.addRect(row,col,f'rgb(255,{r_col},{r_col})')

In [55]:
def run():
    grid = []
    distance = []
    population = []
    
    # Initialize grid and population
    init_grid(grid, distance, GRID_SIZE)
    init_population(population, grid)
    output_display = display(HTML("<div id='sim-container'>Loading...</div>"), display_id=True)

    
    svg = SVGCanvas(320,320,GRID_SIZE)
    # 1. Create the output ONCE and give it a permanent ID
    
    for t in range(iterations.value):
        migrate(grid, population,distance,DESTINATION)
        svg.clear()
        draw_grid(svg,grid,population)
        new_svg = svg.getCanvas()
        new_svg += f'<p>Iterations: {t}</p>'
        new_svg += f'<p style="color:green">Green indicates no occupants</p>'
        output_display.update(HTML(f"<div id='sim-container'>{new_svg}</div>"))
        time.sleep(sleep.value)

In [57]:

iterations = widgets.IntSlider(value=50, min=1, max=MAX_ITERATIONS, description='Iterations:', style={'description_width': 'initial'})
sleep = widgets.FloatSlider(value=0.1, min=0.0000001, max=1.0, description='Sleep time:', style={'description_width': 'initial'})
display(widgets.VBox([iterations, sleep]))


In [58]:
run()